# 02 — Avaliando o modelo (teste de realidade)

Prever o futuro não pode ser conferido na hora. A saída é **esconder o final do histórico**, prever só com o que sobrou e comparar com o que de fato aconteceu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from callcenter_lab.loading import detect_columns, prepare_series, resample_series

# Para usar um dataset público, rode antes: python scripts/baixar_datasets.py technion-bank
# e troque o caminho abaixo por ../data/raw/technion-bank/technion-bank.csv
csv_path = Path('..') / 'data' / 'samples' / 'chamadas_sinteticas.csv'
df = pd.read_csv(csv_path)
df.head()
date_col, value_col = detect_columns(df)
series = resample_series(prepare_series(df, date_col, value_col), '30min', how='sum', fill='zero')

from callcenter_lab.backtest import run_backtest
from callcenter_lab.model import forecast, load_forecaster

model = load_forecaster()

## Rodar o teste

Escondemos o último dia (48 intervalos). A referência simples é a **mesma hora da semana anterior**: `season=336` (7 dias × 48 intervalos).

In [ ]:
horizon = 48
bt = run_backtest(series, horizon, lambda hist, n: forecast(model, hist, n), season=336)

print(f'WAPE TimesFM:              {bt.wape:.1f}%')
print(f'WAPE mesma semana anterior: {bt.baseline_wape:.1f}%')
print(f'Viés:                       {bt.bias_pct:+.1f}%  (positivo = previu chamadas a mais)')
print(f'MAE:                        {bt.mae:.1f} chamadas por intervalo')
print(f'Cobertura da faixa:         {bt.coverage:.0%} (ideal: perto de 80%)')

## Como ler

- **WAPE**: erro total dividido pelo total de chamadas. Menor é melhor. Não se distorce com horários de pouco movimento ou fechados.
- Se o WAPE do TimesFM é menor que o de *mesma semana anterior*, o modelo agrega valor.
- **Viés**: positivo significa que a previsão pediu mais chamadas do que chegaram (risco de sobra de agentes); negativo, o contrário (risco de falta).
- **Cobertura** bem abaixo de 80% significa que a faixa de incerteza é otimista demais.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
bt.history.iloc[-48 * 3:].plot(ax=ax, color='gray', label='Histórico')
bt.actual.plot(ax=ax, color='black', label='Real')
bt.baseline.plot(ax=ax, color='lightgray', ls=':', label='Mesma semana anterior')
bt.point.plot(ax=ax, color='tab:blue', lw=2, label='TimesFM')
ax.fill_between(bt.point.index, bt.lower, bt.upper, alpha=0.2)
ax.set_ylabel('Chamadas')
ax.legend()
plt.show()

## Erro por horário do dia

Onde o modelo erra mais: nos picos ou fora deles?

In [ ]:
errors = (bt.actual - bt.point).abs()
errors.groupby(errors.index.hour).mean().plot(kind='bar', figsize=(10, 3), title='Erro médio por hora do dia (chamadas por intervalo)')
plt.show()

## Para experimentar

- Aumente o `horizon` para 336 (uma semana inteira).
- Rode o mesmo teste com um dataset público. Onde o erro cresce? Em dias atípicos, como feriados?